# Sigma‑Hole Docking Pipeline – Google Colab

This notebook lets you run the sigma‑hole docking pipeline on your own data (or the example data included in the repo).
It handles:
1️⃣ Installing required packages
2️⃣ Uploading your files (ligand structures, receptor, MO surface‑analysis files, or a ready‑made CSV)
3️⃣ Extracting Vmax values from `*_surfanalysis.txt` files if you provide them
4️⃣ Building an input CSV (compound_id, smiles, halogen, vmax) – you can edit smiles/halogen after generation
5️⃣ Running the docking pipeline and visualising results

**Tip:** If you already have a CSV with the columns `compound_id,smiles,halogen,vmax` and the corresponding ligand structure files (`.sdf` or `.pdb`), just upload them and skip the Vmax‑extraction step.


In [ ]:
# Install dependencies from the project's requirements file
import os, subprocess, sys
req_path = 'requirements_colab.txt'
if os.path.exists(req_path):
    print(f'Installing from {req_path}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', req_path])
else:
    print('requirements_colab.txt not found – installing core packages')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'numpy', 'pandas', 'rdkit', 'matplotlib', 'seaborn',
                           'py3Dmol', 'pillow', 'pydantic'])


## 📂 Check what’s already in the notebook’s working directory
(If you launched the notebook from the repo folder, you should see the source files.)

In [ ]:
import os
files = [f for f in sorted(os.listdir('.')) if not f.startswith('.')]
print('Files in current directory:')
for f in files:
    print(f'  {f}')


## 📤 Upload your data
You can upload any combination of the following:
- Ligand structure files (`.sdf`, `.pdb`, `.mol2`)
- Receptor file (`.pdbqt`)
- MO surface‑analysis files (`*_surfanalysis.txt` and/or the accompanying `.pdb`)
- A ready‑made input CSV with columns `compound_id,smiles,halogen,vmax`

After uploading, the files will be available in the current working directory.

In [ ]:
from google.colab import files
print('Select files to upload (you can select multiple):')
uploaded = files.upload()
for name in uploaded.keys():
    print(f'  → {name}')


## 🔧 Helper: Extract Vmax from `*_surfanalysis.txt` files
The function below reads a surface‑analysis text file, finds the **surface maxima** section, and returns the largest Vmax value (in kcal mol⁻¹).

In [ ]:
import re
def extract_vmax_from_txt(txt_path):
    """Return the largest Vmax (kcal/mol) from a *_surfanalysis.txt file."""
    with open(txt_path, 'r', encoding='utf-8') as f:
        content = f.read()
    # Locate the 'Number of surface maxima:' line and capture everything until the next blank line
    match = re.search(r'Number of surface maxima:\s*(\d+)\s*\n([\s\S]*?)(?:\n\n|$)', content)
    if not match:
        raise ValueError(f'Could not find maxima section in {txt_path}')
    lines = match.group(2).strip().splitlines()
    # The first line after the header corresponds to the * entry (largest maximum)
    first = lines[0].strip()
    # Columns: #   a.u.   eV   kcal/mol   X/Y/Z
    # The kcal/mol value is the 4th column (index 3)
    try:
        vmax_kcalmol = float(first.split()[3])
    except (IndexError, ValueError) as e:
        raise ValueError(f'Could not parse Vmax from line: {first}') from e
    return vmax_kcalmol

# Quick test (if a sample file exists)
sample = 'MO-1-SHD_surfanalysis.txt'
if os.path.exists(sample):
    try:
        v = extract_vmax_from_txt(sample)
        print(f'Example Vmax from {sample}: {v:.2f} kcal/mol')
    except Exception as e:
        print(f'Could not read example: {e}')
else:
    print('No example MO file found in the directory.')


## 🧪 Build input CSV from uploaded MO files (optional)
If you uploaded one or more `*_surfanalysis.txt` files, run the cell below to automatically create a DataFrame with:
- `compound_id` – filename without the `_surfanalysis.txt` suffix
- `vmax` – extracted Vmax (kcal/mol)
- `smiles` and `halogen` – left as empty strings for you to fill in (you can edit the table after it’s displayed).

If you already have a CSV, you can skip this step and upload it directly.

In [ ]:
import pandas as pd

def make_csv_from_mo_files():
    """Scan the directory for *_surfanalysis.txt files and return a DataFrame ready for the pipeline."""
    records = []
    for fname in os.listdir('.'):
        if fname.endswith('_surfanalysis.txt'):
            compound = fname.replace('_surfanalysis.txt', '')
            try:
                vmax = extract_vmax_from_txt(fname)
            except Exception as e:
                print(f'⚠️  Skipping {fname}: {e}')
                continue
            records.append({
                'compound_id': compound,
                'smiles': '',          # to be filled by user
                'halogen': '',         # to be filled by user
                'vmax': vmax
            })
    if not records:
        return None
    df = pd.DataFrame(records, columns=['compound_id','smiles','halogen','vmax'])
    return df

mo_df = make_csv_from_mo_files()
if mo_df is not None:
    print(f'Found {len(mo_df)} MO file(s). Generated preview:')
    display(mo_df)
    # Let the user edit smiles/halogen if needed
    print('\nEdit the `smiles` and `halogen` columns below as needed, then run the next cell to save the CSV.')
else:
    print('No *_surfanalysis.txt files found – you will need to upload your own input CSV.')


In [ ]:
# If you have a DataFrame from the previous cell (mo_df), you can edit it here.
# Otherwise, upload a CSV now.
if 'mo_df' in locals() and mo_df is not None:
    # Show an editable dataframe (requires ipywidgets)
    try:
        from IPython.display import display
        import ipywidgets as widgets
        # Create a widget that allows editing
        df_widget = widgets.DataFrame(mo_df)
        display(df_widget)
        # On button click, capture the edited dataframe
        def on_button_clicked(b):
            global edited_df
            edited_df = df_widget.value
            print('Edited dataframe captured.')
        button = widgets.Button(description='Save edits')
        button.on_click(on_button_clicked)
        display(button)
        print('Click the button after editing, then run the next cell to save the CSV.')
    except Exception as e:
        # Fallback: just show the dataframe and tell user to copy‑paste
        print('Unable to create interactive widget (', e, ').')
        print('Please manually edit the values if needed and run the next cell to save as CSV.')
        edited_df = mo_df
else:
    print('Please upload your input CSV now (columns: compound_id,smiles,halogen,vmax).')
    uploaded2 = files.upload()
    for name in uploaded2.keys():
        if name.endswith('.csv'):
            print(f'Uploaded CSV: {name}')
            edited_df = pd.read_csv(name)
            break
    else:
        raise FileNotFoundError('No CSV file was uploaded.')


In [ ]:
# Save the (possibly edited) dataframe as the input CSV for the pipeline
if 'edited_df' in locals():
    csv_path = 'input_from_upload.csv'
    edited_df.to_csv(csv_path, index=False)
    print(f'Saved input CSV to `{csv_path}` with {len(edited_df)} rows.')
    print(edited_df.head())
else:
    print('No dataframe to save – check previous steps.')


## 🧬 Select receptor file
You can either:
- Use the receptor already present in the directory (`receptor.pdbqt` by default),
- Or upload your own receptor file (`.pdbqt`).
The cell below will let you upload a receptor; if you upload one, it will be used, otherwise it falls back to `receptor.pdbqt` if it exists.

In [ ]:
print('Upload a receptor file (PDBQT) if you want to override the default.')
receptor_upload = files.upload()
receptor_path = 'receptor.pdbqt'  # default
if receptor_upload:
    # Assume the first uploaded file is the receptor
    uploaded_name = list(receptor_upload.keys())[0]
    receptor_path = uploaded_name
    print(f'Using uploaded receptor: {receptor_path}')
elif os.path.exists(receptor_path):
    print(f'Using existing receptor: {receptor_path}')
else:
    print(f'⚠️  No receptor file found at {receptor_path}. Please upload one.')


## 🚀 Run the sigma‑hole docking pipeline
Now we import the pipeline, point it to the input CSV and receptor, and execute.

In [ ]:
# Make sure the current directory is on Python path so we can import sigma_hole_pipeline
import sys
if '.' not in sys.path:
    sys.path.insert(0, '.')

from sigma_hole_pipeline import SigmaHolePipeline

# Paths – adjust if you used different names
input_csv = 'input_from_upload.csv'   # from previous step
receptor_input = receptor_path        # selected above
structure_dir = '.'                   # look for ligand structures in the current folder
structure_ext = '.sdf'                # change if you have .pdb or other formats

print(f'Input CSV: {input_csv}')
print(f'Receptor:   {receptor_input}')
print(f'Structure dir: {structure_dir} (extension {structure_ext})')

# Quick sanity check
if not os.path.exists(input_csv):
    raise FileNotFoundError(f'Input CSV not found: {input_csv}')
if not os.path.exists(receptor_input):
    raise FileNotFoundError(f'Receptor not found: {receptor_input}')

pipeline = SigmaHolePipeline()
print('Starting sigma‑hole pipeline...')
results = pipeline.run_full_pipeline(
    input_csv=input_csv,
    receptor_input=receptor_input,
    structure_dir=structure_dir,
    structure_ext=structure_ext
)
print('Pipeline completed!')
print(f'Result keys: {list(results.keys())}')


## 📊 Access and visualise results
The pipeline returns a dictionary. We’ll display the docking results table, some analysis summaries, and plots.

In [ ]:
import pandas as pd

# --- Docking results ---
if 'docking_results' in results:
    df_dock = results['docking_results']
    print('Docking Results (first 10 rows):')
    display(df_dock.head(10))
    print(f'Total compounds docked: {len(df_dock)}')
else:
    print('No docking results found.')

# --- Analysis results ---
if 'analysis' in results:
    analysis = results['analysis']
    print('\nAnalysis summaries:')
    for key, val in analysis.items():
        if isinstance(val, pd.DataFrame):
            print(f'--- {key} --- ({val.shape[0]} rows)')
            display(val.head())
        else:
            print(f'{key}: {val}')
else:
    print('No analysis section in results.')


### 📈 Binding‑energy distribution
If docking results are present, we plot a histogram of the binding energies.

In [ ]:
if 'docking_results' in results and len(results['docking_results']) > 0:
    import matplotlib.pyplot as plt
    import seaborn as sns
    df = results['docking_results']
    plt.figure(figsize=(8,5))
    sns.histplot(data=df, x='binding_energy_kcalmol', bins=20, color='#4c72b0')
    plt.title('Binding‑energy distribution')
    plt.xlabel('Binding Energy (kcal/mol)')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('Skipping plot – no docking results.')


### 🏆 Top hits
If the analysis contains a `top_hits` DataFrame, we show it.

In [ ]:
if 'analysis' in results and 'top_hits' in results['analysis']:
    top = results['analysis']['top_hits']
    print('Top 5 hits:')
    display(top.head())
else:
    print('No top_hits table found in analysis.')


## 💾 Save results for download
All relevant DataFrames are saved as CSV files in the current directory. You can download them from the file pane on the left.

In [ ]:
if 'docking_results' in results:
    results['docking_results'].to_csv('docking_results.csv', index=False)
    print('Saved docking_results.csv')

if 'analysis' in results:
    for key, val in results['analysis'].items():
        if isinstance(val, pd.DataFrame):
            val.to_csv(f'analysis_{key}.csv', index=False)
            print(f'Saved analysis_{key}.csv')
else:
    print('Nothing to save.')


## 🎉 That’s it!
You have:
1. Uploaded your data (or used the MO files to generate Vmax)
2. Selected a receptor
3. Run the sigma‑hole docking pipeline
4. Visualised and saved the results.

Feel free to tweak the parameters (e.g., change `structure_ext` to `.pdb` if you uploaded PDB ligand files) and re‑run the pipeline from the **Run the sigma‑hole docking pipeline** cell onward.

---
*Built with the sigma‑hole docking pipeline. For questions or issues, consult the repository’s README.*
